# Monitoring& Observability

**Module 12 · Lesson 12.8**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why AI Observability Is Different
- The Three Pillars (and a Fourth)
- Traces and Spans
- OpenTelemetry and the gen_ai Conventions
- The Golden Signals
- Alerting on SLOs and Anomalies

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## All green, and the users were right anyway

> 💡 **Info**
>
> The support tickets kept saying the same thing: the assistant “felt off” this week. So you did the responsible thing and opened every dashboard. **p95 latency: fine. Error rate: zero. HTTP 200 on every request.** All green, top to bottom. And yet the users were not imagining it. Someone had shipped a quiet model swap, and since then the answers had gotten vaguer, the tokens per reply had crept up so the bill was climbing, and the time to the first word had **tripled**. Your monitoring could not see a single bit of it — because it was built to watch a *normal* web app, and a normal web app only fails in three ways: it goes down, it gets slow, or it throws errors. An AI app fails in a fourth way that those dashboards have no gauge for: it gets **worse**. This lesson builds observability that can actually see an AI app — traces that show where each request spent its time, OpenTelemetry so you instrument once and switch backends freely, the golden signals that matter for LLMs, alerting on the symptoms users feel, and online evaluation that catches a quality drift before the tickets arrive — and you can run every piece with no backend and no API key.

### 🎯 What you will be able to do after this lesson

- **Build** LLM observability — traces of a request’s spans, OpenTelemetry `gen_ai` instrumentation, a golden-signals dashboard, SLO and anomaly alerts, and an online-eval loop — as runnable models plus an illustrative OTel + Langfuse setup.

- **Compare** logs, metrics, and traces (which question each answers) and monitoring vs observability.

- **Operate** a golden-signals dashboard, an SLO with an error budget, and a drift and PII-redaction loop on live traffic.

- **Evaluate** an observability setup: does it capture traces, use OTel, track the AI signals, alert on symptoms, and watch quality in production?

> 📦 **Info**
>
> ✅ Before you startIn **12.6** you gated a merge on an eval that scored the model’s answers — that same eval, run on *live* traffic, is how you watch quality in production. In **12.7** the autoscaler read concurrency and queue depth — monitoring is where those scaling signals live and how you see them. In **12.3** the gateway metered tokens per team — observability turns that into a cost dashboard. Building the eval set you score against is **Lesson 14.4**; diagnosing an outage with this telemetry is **Lesson 12.9**; cutting the bill it surfaces is **Module 13**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🩺 **Analogy**
>
> Monitoring an AI app is being the **doctor at a check-up, not the security guard at the door**. A guard answers one yes/no question: is the patient in the building and moving? (Is the service up, fast, not erroring?) A doctor asks a deeper one: *how are they actually doing?* — and to answer it uses three different instruments. A **symptom log** (what happened, and when), **vital-sign charts** that trend over time (is the fever rising?), and a **full scan** that traces a problem to the exact organ. Any one alone misleads; together they explain. **Where it breaks down:** a doctor examines one patient at a time, while your telemetry watches millions of requests at once — which is exactly why it must sample and summarize rather than read every chart by hand.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Green dashboards mean the AI app is healthy.”** Latency and error rate can be perfect while quality regressed, cost doubled, and time-to-first-token tripled. You have to instrument the things users actually feel.
> - **“Observability is just logging.”** Logs are one pillar. You also need **metrics** (a number over time), **traces** (the causal chain of one request), and **evals** (was the answer any good?). Different questions, different pillars.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that will bite. **Wiring your app to one platform’s proprietary SDK** — when you want to switch tools, you re-instrument every call; use OpenTelemetry’s vendor-neutral `gen_ai` conventions instead. And **capturing raw prompts and responses straight into your traces** — that quietly ships user emails, phone numbers, and secrets into a third-party backend; redact PII *before* it enters a span. (And alerting on every blip until on-call mutes the pager — more on that in Step 6.)

---

## 🎯 Concept 1: Why AI Observability Is Different

### Why AI Observability Is Different

A normal web app fails by going down, slow, or erroring - three things ops dashboards watch. An AI app also fails by getting worse: quality drops, cost climbs, time-to-first-token grows, while every ops signal stays green.

Start with why the normal playbook goes blind. Traditional monitoring answers three questions: **is it up, is it fast, is it erroring?** Those are the right questions for a normal web app, because that is how a normal web app fails. An AI app can pass all three and still be failing its users, because it has a whole class of failure the ops dashboard has no gauge for. The model got swapped and answers got **vaguer** (quality dropped). A longer system prompt means every reply burns more tokens (cost climbed). A reasoning model streams its first token far later (time-to-first-token grew). None of that moves p95 latency, the error rate, or the HTTP status — so the dashboard stays green while users quietly suffer. This is the difference between **monitoring** (is a known thing broken?) and **observability** (why does the system behave the way it does, including questions you did not think to ask in advance?). An AI app needs the second kind. The block shows the same deploy through both lenses, keyless.

> 🚗 **Analogy**
>
> It is the **green dashboard and the strange engine noise**. Your car’s dashboard has a fixed set of lights: fuel, temperature, oil, check-engine. They are all off, so by the dashboard the car is perfect. But you can hear a new rattle, the steering pulls slightly, and the ride feels rougher than last week — real problems the dashboard was simply never built to show. The ops dashboard is those fixed warning lights; observability is the mechanic who listens to the engine, reads the trip computer, and traces the rattle to its source.

A model swap ships. p95 latency, error rate, and HTTP status are all unchanged, but users say answers got worse. What does the ops dashboard show?

**📝 `01_why_different.py`** - *runnable*

In [ ]:
# The ops dashboard watches a NORMAL web app: is it up, fast, erroring?
# An AI app can be perfect on all three while the things users feel quietly rot.
BEFORE = {"p95_ms": 2100, "error_rate": 0.000, "http_ok": 1.00, "quality": 8.4, "cost_per_reply": 0.011, "ttft_ms": 700}
AFTER  = {"p95_ms": 2160, "error_rate": 0.000, "http_ok": 1.00, "quality": 6.9, "cost_per_reply": 0.019, "ttft_ms": 2100}

def health(v, ok):
    return "green" if ok else "RED"

print("A model swap shipped. What each dashboard sees:")
print()
print("  OPS dashboard (built for a normal web app):")
print("    p95 latency   {:>5}ms -> {:>5}ms   {}".format(BEFORE["p95_ms"], AFTER["p95_ms"], health(0, True)))
print("    error rate    {:>5.0%} -> {:>5.0%}   {}".format(BEFORE["error_rate"], AFTER["error_rate"], health(0, True)))
print("    HTTP 200 rate {:>5.0%} -> {:>5.0%}   {}".format(BEFORE["http_ok"], AFTER["http_ok"], health(0, True)))
print("    => all green. On-call sees nothing.")
print()
print("  AI signals (what the users actually feel):")
print("    answer quality {:>4.1f} -> {:>4.1f}   {}".format(BEFORE["quality"], AFTER["quality"], health(0, False)))
print("    cost / reply  ${:>5.3f} -> ${:>5.3f}  {}".format(BEFORE["cost_per_reply"], AFTER["cost_per_reply"], health(0, False)))
print("    time-to-first-token {:>4}ms -> {:>4}ms  {}".format(BEFORE["ttft_ms"], AFTER["ttft_ms"], health(0, False)))
print("    => quality fell, cost nearly doubled, first word takes 3x longer.")
print()
print("Monitoring answers 'is it broken?'. Observability answers 'why does it feel worse?'.")
print("An AI app needs the second kind: instrument quality, cost, and TTFT, not just up/fast/errors.")

# Output:
# A model swap shipped. What each dashboard sees:
#
#   OPS dashboard (built for a normal web app):
#     p95 latency    2100ms ->  2160ms   green
#     error rate       0% ->    0%   green
#     HTTP 200 rate  100% ->  100%   green
#     => all green. On-call sees nothing.
#
#   AI signals (what the users actually feel):
#     answer quality  8.4 ->  6.9   RED
#     cost / reply  $0.011 -> $0.019  RED
#     time-to-first-token  700ms -> 2100ms  RED
#     => quality fell, cost nearly doubled, first word takes 3x longer.
#
# Monitoring answers 'is it broken?'. Observability answers 'why does it feel worse?'.
# An AI app needs the second kind: instrument quality, cost, and TTFT, not just up/fast/errors.

- The **ops dashboard** reads p95 latency, error rate, and HTTP status — all essentially unchanged, all **green**.
- The **AI signals** tell the real story: answer quality fell, cost per reply nearly doubled, and time-to-first-token roughly tripled.
- A normal web app fails by going down, slow, or erroring; an AI app also fails by getting **worse** — a failure mode ops monitoring has no gauge for.
- Monitoring asks “is it broken?”; observability asks “why does it feel worse?” — and that is the kind an AI app needs.

#### 💡 What just happened

⚡ What just happened? A normal web app fails three ways ops dashboards watch: down, slow, erroring. An AI app adds a fourth - it gets worse (quality, cost, TTFT) while every ops signal stays green. Tradeoff / the framing for the lesson: you must instrument the AI-specific signals, not just up/fast/errors. Every later step builds a piece of that instrumentation.

- Three ops gauges (latency, errors, HTTP) sit green while three AI gauges (quality, cost, TTFT) flash red
- Toggle the model deploy to watch the ops panel stay green and the AI panel break

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Three Pillars (and a Fourth)

### The Three Pillars (and a Fourth)

Logs, metrics, and traces each answer a different question. A log is one event’s fields; a metric is a number over time; a trace is the causal chain of one request. Evals add a fourth pillar: was the answer any good?

Observability rests on three **pillars**, and the trick is knowing which question each one answers. A **log** is a record of a single event with its fields — it answers *“what exactly happened in this one event?”* A **metric** is a number aggregated over time — p95 latency, requests per minute, error rate — it answers *“how is this trending across all requests?”* A **trace** follows one request through the system as a chain of timed **spans** — it answers *“where did this particular request spend its time, and why?”* Reach for the wrong pillar and you will stare at the right data asking a question it cannot answer: a metric will show you that p95 spiked but never *which* request or why; a log will show you the event’s fields but not the breakdown inside it. For an AI app there is a fourth pillar the classic three do not cover — **evals**, the measured *quality* of the answer — which we build in Step 7. The block shows one slow request through all three, keyless.

> 👨‍⚕️ **Analogy**
>
> It is the doctor’s three instruments from the warm-up, made concrete. The **symptom log** is the chart note: “10:42, patient reported dizziness, temp 38.1” — one event, fully detailed. The **vital-sign monitor** is the trend line: heart rate over the last hour, telling you it is climbing but not why. The **full-body scan** is the trace: it follows the problem through the body and points at the exact organ. A doctor who only had the trend line would know something is wrong but never what; only the scan localizes it.

Your p95 latency metric spiked at 2pm. You want to know which request caused it and where its time went. Which pillar do you reach for?

**📝 `02_three_pillars.py`** - *runnable*

In [ ]:
# One slow request, seen through the three pillars - each answers a DIFFERENT question.
# A LOG is one event with fields. A METRIC is a number over time. A TRACE is the causal chain.
request = {
    "id": "req-42", "route": "/chat", "user": "u_88", "status": 200,
    "model": "claude-sonnet-4-6", "latency_ms": 6100, "input_tokens": 1800, "output_tokens": 240,
}
p95_series = [2000, 2100, 2050, 2200, 6100]   # last 5 minutes of p95, this request is the spike
spans = [("retrieve_context", 200), ("llm_generate", 5800), ("format_reply", 40)]

print("PILLAR 1 - LOG  (question: what exactly happened in THIS event?)")
for k, v in request.items():
    print("    {:<14} {}".format(k, v))
print()
print("PILLAR 2 - METRIC  (question: how is p95 latency TRENDING?)")
print("    p95 over the last 5 minutes: " + " -> ".join("{}ms".format(x) for x in p95_series))
print("    the trend shows a spike; it does NOT tell you which request or why.")
print()
print("PILLAR 3 - TRACE  (question: WHERE did this one request spend its time?)")
total = sum(ms for _, ms in spans)
for name, ms in spans:
    print("    {:<16} {:>5}ms  ({:>2.0f}% of the request)".format(name, ms, 100 * ms / total))
print("    the trace pins the cost to llm_generate - the log and metric could not.")
print()
print("Logs, metrics, traces answer different questions; evals add a fourth: was the ANSWER any good?")

# Output:
# PILLAR 1 - LOG  (question: what exactly happened in THIS event?)
#     id             req-42
#     route          /chat
#     user           u_88
#     status         200
#     model          claude-sonnet-4-6
#     latency_ms     6100
#     input_tokens   1800
#     output_tokens  240
#
# PILLAR 2 - METRIC  (question: how is p95 latency TRENDING?)
#     p95 over the last 5 minutes: 2000ms -> 2100ms -> 2050ms -> 2200ms -> 6100ms
#     the trend shows a spike; it does NOT tell you which request or why.
#
# PILLAR 3 - TRACE  (question: WHERE did this one request spend its time?)
#     retrieve_context   200ms  ( 3% of the request)
#     llm_generate      5800ms  (96% of the request)
#     format_reply        40ms  ( 1% of the request)
#     the trace pins the cost to llm_generate - the log and metric could not.
#
# Logs, metrics, traces answer different questions; evals add a fourth: was the ANSWER any good?

- **Pillar 1, the log**: every field of one event (route, user, model, tokens) — it answers *what happened in this event*.
- **Pillar 2, the metric**: p95 latency over the last few minutes — it shows the *trend and the spike*, but not which request or why.
- **Pillar 3, the trace**: the same request split into spans, pinning the cost to `llm_generate` — the log and metric could not do that.
- Different questions, different pillars — and a fourth, **evals**, answers the one none of these do: *was the answer any good?*

#### 💡 What just happened

⚡ What just happened? Logs (one event’s fields), metrics (a number over time), and traces (the causal chain of one request) each answer a different question; evals add a fourth - answer quality. Tradeoff: you need all of them, because reaching for the wrong pillar leaves you staring at data that cannot answer your question. Next: the pillar that is the killer feature for an AI app - the trace.

- One slow request shown as a LOG (its fields), a METRIC (p95 over time), and a TRACE (the span chain)
- Switch tabs to see each pillar answer a different question, plus an EVAL score

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Traces and Spans

### Traces and Spans

A trace is a tree of spans, each a timed unit of work with a parent. Rebuilding the waterfall for one request shows where the time actually went - and for an LLM app, the model span usually dominates, with time-to-first-token the number users feel.

The trace is the pillar that earns its keep in an AI app, so it gets its own step. A **trace** represents one request as a tree of **spans**. Each span is a single unit of work — a retrieval, a model call, a tool call, a bit of post-processing — with a name, a start time, a duration, and a parent span, so the spans nest to show what happened inside what. Drawn as a **waterfall**, the trace makes the shape of a request obvious at a glance: which steps ran in sequence, which nested inside others, and above all *where the time went*. For an LLM app the answer is almost always the same — the **model-generation span dominates**, dwarfing your own code — which tells you the wins are in the model call (streaming, a smaller model, caching), not your glue code. A trace also surfaces the latency that decides how responsive the app feels: **time-to-first-token** (TTFT), the delay before the model streams its first token. Here TTFT is measured *inside* the model span, so the full blank-screen wait a user feels is the time to reach the model *plus* that TTFT. A single end-to-end latency figure hides all of this; the span tree shows it. The block rebuilds a waterfall, keyless.

> 🏁 **Analogy**
>
> It is a **race split into timed segments**. “She finished in six minutes” tells you almost nothing about how to coach her. Split the race into its legs — the start, the straight, the hill, the sprint — and time each one, and the story jumps out: nearly all the time was lost on the hill. That is a trace. Each leg is a span; the segment times are the durations; and the split instantly shows you the one place worth training, instead of shaving milliseconds off the parts that were already fast.

A trace shows a request took a few seconds, and the llm_generate span was almost all of it. Where should you optimize?

**📝 `03_traces_spans.py`** - *runnable*

In [ ]:
# A TRACE is a tree of SPANS. Each span is one unit of work with a start, a duration, and a parent.
# Rebuild the waterfall for one request and find where the time went.
# (name, parent, start_ms, duration_ms)
spans = [
    ("chat_request",   None,           0, 6130),
    ("retrieve",       "chat_request", 20, 200),
    ("llm_generate",   "chat_request", 230, 5800),
    ("tool_call",      "chat_request", 6030, 60),
    ("format_reply",   "chat_request", 6090, 40),
]
TTFT_MS = 800   # first streamed token arrived 800ms into llm_generate

root_dur = spans[0][3]
scale = 48.0 / root_dur   # 48 chars = the whole request
print("Trace for chat_request (one user message), total {}ms:".format(root_dur))
print()
for name, parent, start, dur in spans:
    depth = 0 if parent is None else 1
    pad = "  " * depth
    lead = int(start * scale)
    bar = "#" * max(1, int(dur * scale))
    pct = 100 * dur / root_dur
    print("  {}{:<14}{} {:<50} {:>5}ms  {:>2.0f}%".format(pad, name, "", " " * lead + bar, dur, pct))
print()
llm_start = next(s for n, p, s, d in spans if n == "llm_generate")
llm = next(d for n, p, s, d in spans if n == "llm_generate")
print("Reading the waterfall:")
print("  llm_generate is {:.0f}% of the request - the model call dominates, not your code.".format(100 * llm / root_dur))
print("  TTFT (first-token latency) was {}ms, measured inside the model span.".format(TTFT_MS))
print("  Blank-screen wait = {}ms to reach the model + {}ms TTFT = {}ms before ANY text appeared.".format(llm_start, TTFT_MS, llm_start + TTFT_MS))
print("  A single end-to-end latency number hid all of this; the span tree shows exactly where to optimize.")

# Output:
# Trace for chat_request (one user message), total 6130ms:
#
#   chat_request   ################################################    6130ms  100%
#     retrieve       #                                                    200ms   3%
#     llm_generate    #############################################      5800ms  95%
#     tool_call                                                     #      60ms   1%
#     format_reply                                                  #      40ms   1%
#
# Reading the waterfall:
#   llm_generate is 95% of the request - the model call dominates, not your code.
#   TTFT (first-token latency) was 800ms, measured inside the model span.
#   Blank-screen wait = 230ms to reach the model + 800ms TTFT = 1030ms before ANY text appeared.
#   A single end-to-end latency number hid all of this; the span tree shows exactly where to optimize.

- The **waterfall** draws one request as nested spans — `retrieve`, `llm_generate`, `tool_call`, `format_reply` — each with its own bar and share.
- The **model span dominates** the request; your own code (retrieval, formatting) is a thin sliver by comparison.
- **TTFT** (time-to-first-token) is the model’s first-token latency; add the time to reach the model and you get the real blank-screen wait a single end-to-end latency number buries.
- A single latency figure hid all of this; the span tree shows exactly where to spend your optimization effort.

#### 💡 What just happened

⚡ What just happened? A trace is a tree of spans - each a timed unit of work with a parent - and its waterfall shows where a request’s time went. For an LLM app the model span dominates, and TTFT is the number users feel. Tradeoff: tracing adds a little per-request overhead and storage, so at production scale you do not keep every trace - you **sample**. **Head-based** sampling decides at the request’s start (cheap, but blind: it can drop the rare slow or failed trace you most wanted); **tail-based** sampling decides after the whole trace is seen (it keeps the interesting ones, at the cost of buffering each trace first). Next: how to emit these spans in a vendor-neutral way.

- A span waterfall for one request: retrieve, llm_generate, tool_call, format_reply
- Hover a span to highlight its slice; the model span dominates and TTFT is marked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: OpenTelemetry and the gen_ai Conventions

### OpenTelemetry and the gen_ai Conventions

Instrument once with OpenTelemetry’s gen_ai semantic conventions and the span is vendor-neutral: the same attributes render in Langfuse, Phoenix, or Datadog. Wire to a proprietary SDK instead and switching tools means re-instrumenting every call.

Now, how do you actually emit those spans without marrying one vendor for life? The answer the industry settled on is **OpenTelemetry** (OTel) — an open standard for producing telemetry — plus its **gen_ai semantic conventions**, an agreed set of attribute names for LLM calls. Instead of every tool inventing its own field for “which model” or “how many input tokens,” OTel names them once: `gen_ai.operation.name`, `gen_ai.system`, `gen_ai.request.model`, `gen_ai.usage.input_tokens` and `output_tokens`, `gen_ai.response.finish_reasons`. The same span format describes a call to any provider — whether `gen_ai.system` records **OpenAI**, **Anthropic**, or **Google**’s Gemini — so one instrumentation covers them all. You create the span *once*, tagged with those standard attributes, and export it over **OTLP** (the OTel wire protocol) to whatever backend you like — **Langfuse**, **Arize Phoenix**, **Datadog** — and each one understands it unchanged. Wire your app to a single vendor’s proprietary SDK instead, and the day you want to switch or add a tool you re-instrument every call site. (The gen_ai conventions are still stabilizing through 2026, so pin your library versions, but the direction is settled.) The block builds one gen_ai span and fans it out, keyless.

> 🔌 **Analogy**
>
> It is a **universal power adapter for your telemetry**. Without it, every backend is a different wall socket, and wiring your app to one vendor is soldering your plug to their outlet — move to another country (another tool) and you rewire the whole appliance. OTel is the standard plug: you fit it once, and the same device works in Langfuse’s socket, Phoenix’s, or Datadog’s without any change to the appliance. The gen_ai conventions are the agreed pin layout, so everyone’s socket accepts the same shape.

You instrument your app with one observability vendor’s proprietary SDK. A year later you want to try a different backend. What does the switch cost?

**📝 `04_otel_genai.py`** - *runnable*

In [ ]:
# Instrument ONCE with OpenTelemetry's gen_ai semantic conventions, export ANYWHERE.
# The span is vendor-neutral: the same attributes render in Langfuse, Phoenix, or Datadog.
def gen_ai_span(model, in_tok, out_tok, ttft_ms, dur_ms, finish):
    # keys follow the OTel gen_ai.* convention (stabilizing in 2026)
    return {
        "name": "chat",
        "kind": "CLIENT",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "anthropic",
        "gen_ai.request.model": model,
        "gen_ai.usage.input_tokens": in_tok,
        "gen_ai.usage.output_tokens": out_tok,
        "gen_ai.response.finish_reasons": [finish],
        "gen_ai.server.time_to_first_token_ms": ttft_ms,
        "duration_ms": dur_ms,
    }

span = gen_ai_span("claude-sonnet-4-6", 1800, 240, 800, 5800, "end_turn")
print("One gen_ai span (created once, in your app):")
for k, v in span.items():
    print("    {:<38} {}".format(k, v))
print()
# The SAME span object is what every backend ingests - no re-instrumentation.
BACKENDS = ["Langfuse", "Arize Phoenix", "Datadog"]
print("Exported unchanged to each backend via OTLP:")
for b in BACKENDS:
    print("    {:<14} reads model={}, tokens={}+{}, ttft={}ms - same span, zero code changes".format(
        b, span["gen_ai.request.model"], span["gen_ai.usage.input_tokens"],
        span["gen_ai.usage.output_tokens"], span["gen_ai.server.time_to_first_token_ms"]))
print()
print("Wire to a vendor's proprietary SDK instead and you re-instrument every call to switch tools.")
print("OTel gen_ai conventions = instrument once, swap backends freely (no lock-in).")

# Output:
# One gen_ai span (created once, in your app):
#     name                                   chat
#     kind                                   CLIENT
#     gen_ai.operation.name                  chat
#     gen_ai.system                          anthropic
#     gen_ai.request.model                   claude-sonnet-4-6
#     gen_ai.usage.input_tokens              1800
#     gen_ai.usage.output_tokens             240
#     gen_ai.response.finish_reasons         ['end_turn']
#     gen_ai.server.time_to_first_token_ms   800
#     duration_ms                            5800
#
# Exported unchanged to each backend via OTLP:
#     Langfuse       reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#     Arize Phoenix  reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#     Datadog        reads model=claude-sonnet-4-6, tokens=1800+240, ttft=800ms - same span, zero code changes
#
# Wire to a vendor's proprietary SDK instead and you re-instrument every call to switch tools.
# OTel gen_ai conventions = instrument once, swap backends freely (no lock-in).

- One **gen_ai span** is built with the standard OTel attributes — operation, system, model, token usage, finish reason, TTFT.
- The *same* span is exported to **Langfuse**, **Arize Phoenix**, and **Datadog** over OTLP — each reads it with **zero code changes**.
- That portability is the whole point: instrument once, swap or fan out backends freely.
- Wire to a proprietary SDK instead and switching tools means re-instrumenting every call — the lock-in OTel exists to prevent.

#### 💡 What just happened

⚡ What just happened? OpenTelemetry’s gen_ai conventions give LLM spans standard attribute names, so you instrument once and export the same span to Langfuse, Phoenix, or Datadog unchanged. Tradeoff: OTel adds a little setup and its gen_ai spec is still stabilizing (pin versions), in exchange for no vendor lock-in. Next: which signals to actually put on the dashboard.

- Assemble one span from gen_ai.* attribute tiles, then fan it out to Langfuse, Phoenix, Datadog unchanged
- Flip to a proprietary SDK to watch every backend demand its own re-instrumentation

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: The Golden Signals

### The Golden Signals

Ops watches latency and errors. An LLM app needs those plus TTFT, throughput, tokens and cost per request, cache hit rate, and a quality score - each sliced by model and team - because cost and quality are exactly what the ops dashboard never shows.

With spans flowing, what goes on the dashboard? Site reliability practice has a classic short list of **golden signals** — latency, traffic, errors, saturation — and for an LLM app you keep those and add the ones that matter here. **Latency**, read at **p95 and p99** (the tail users actually feel, not the mean). **Time-to-first-token**, the blank-screen wait. **Error rate** and **throughput** (requests per minute). **Tokens and cost per request** — the two numbers a normal dashboard never shows, and the ones that quietly blow the budget. **Cache hit rate**, since a cached reply is nearly free (from 12.4). And a **quality score** — the one signal that tells you whether answers are actually good, built in Step 7. The move that makes this powerful is **slicing**: the same signals broken down *by model and by team*, so a regression shows up as “the new model on the support team” rather than a vague global dip. Read the mean and you will miss the tail; read p95 and you see what users feel. The block computes the golden signals from a window of requests, keyless.

> ✈️ **Analogy**
>
> It is a **cockpit, not a single fuel gauge**. A pilot does not fly on “is the engine on?” alone; the panel shows airspeed, altitude, heading, fuel, and engine temperature together, because each catches a different way the flight can go wrong. A dashboard with only latency and errors is a cockpit with one gauge — it flies fine until the thing it does not measure kills you. The golden signals are the full panel: latency and TTFT for speed, tokens and cost for fuel burn, quality for whether you are still on course.

Your dashboard shows only average latency and error rate. Which real problem can slip past it entirely?

**📝 `05_golden_signals.py`** - *runnable*

In [ ]:
# The golden signals for an LLM app, computed from a window of request records.
# Ops watches latency + errors; an AI app ALSO needs TTFT, tokens, cost, cache, and quality.
# (latency_ms, ttft_ms, ok, in_tok, out_tok, cache_hit, quality)
W = [
    (2100, 700, True,  1800, 240, False, 8.6),
    (2400, 820, True,  1500, 300, True,  8.1),
    (6100, 2100, True, 2000, 260, False, 6.9),
    (2200, 690, True,  1600, 210, True,  8.4),
    (2000, 640, False, 1700, 0,   False, 0.0),   # a failed request
    (2600, 900, True,  1900, 280, False, 8.0),
]
PRICE_IN, PRICE_OUT = 3.0 / 1_000_000, 15.0 / 1_000_000   # $/token (illustrative)

def pctl(values, p):
    s = sorted(values); k = max(0, int(round((p / 100.0) * (len(s) - 1))))
    return s[k]

ok = [r for r in W if r[2]]
lat = [r[0] for r in W]
cost = sum(r[3] * PRICE_IN + r[4] * PRICE_OUT for r in W)
print("Golden signals over a {}-request window:".format(len(W)))
print("  latency  p50 {}ms   p95 {}ms".format(pctl(lat, 50), pctl(lat, 95)))
print("  TTFT     p50 {}ms   p95 {}ms".format(pctl([r[1] for r in ok], 50), pctl([r[1] for r in ok], 95)))
print("  error rate       {:.0%}   ({} of {} failed)".format(1 - len(ok) / len(W), len(W) - len(ok), len(W)))
print("  throughput       {} requests in the window".format(len(W)))
print("  tokens           {} in / {} out".format(sum(r[3] for r in W), sum(r[4] for r in W)))
print("  cost             ${:.4f} total  (${:.4f} per request)".format(cost, cost / len(W)))
print("  cache hit rate   {:.0%}".format(sum(1 for r in W if r[5]) / len(W)))
print("  quality (mean)   {:.1f} / 10   (scored answers only)".format(sum(r[6] for r in ok) / len(ok)))
print()
print("Same signals, sliced by model or team, are your dashboard. p95 (not the mean) is what users feel;")
print("TTFT is what they SEE first; cost and quality are the two ops dashboards never show.")

# Output:
# Golden signals over a 6-request window:
#   latency  p50 2200ms   p95 6100ms
#   TTFT     p50 820ms   p95 2100ms
#   error rate       17%   (1 of 6 failed)
#   throughput       6 requests in the window
#   tokens           10500 in / 1290 out
#   cost             $0.0509 total  ($0.0085 per request)
#   cache hit rate   33%
#   quality (mean)   8.0 / 10   (scored answers only)
#
# Same signals, sliced by model or team, are your dashboard. p95 (not the mean) is what users feel;
# TTFT is what they SEE first; cost and quality are the two ops dashboards never show.

- **Latency at p95** (not the mean) is what users feel; the window’s tail is far worse than its average.
- **TTFT, error rate, and throughput** come straight from the request records — the classic signals, kept.
- **Tokens, cost per request, and cache hit rate** are the AI-specific additions — the numbers an ops dashboard never shows.
- A **quality score** (scored answers only) is the last signal; sliced by model or team, these signals become your dashboard.

#### 💡 What just happened

⚡ What just happened? The golden signals for an LLM app are latency (p95/p99), TTFT, error rate, throughput, tokens and cost per request, cache hit rate, and quality - each sliced by model and team. Tradeoff: more signals cost more to collect and display, but cost and quality are exactly the failures a latency/error dashboard misses. Next: how to alert on these without waking on-call for nothing.

- A cockpit of six tiles (p95 latency, TTFT, error rate, tokens+cost, cache hit, quality) recomputing as requests stream in
- Filter by model or team to see a regression localize

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Alerting on SLOs and Anomalies

### Alerting on SLOs and Anomalies

Turn signals into alerts against an SLO with an error budget, and page only on a sustained breach. A brief self-healing blip should be logged, not paged; anomaly rules catch cost and quality drifts a static threshold would miss.

Signals are only useful if the right one wakes the right person at the right time — and the wrong alerting is worse than none. Anchor alerts to an **SLO** (a service-level objective, e.g. “p95 under three seconds”) with an **error budget** (the small fraction of requests you allow to miss it). Then two rules make alerting sane. First, **page on a sustained breach, not a blip**: require several consecutive bad windows before paging, so a momentary spike that heals itself is logged quietly rather than paging on-call at 3am. Second, add **anomaly** rules for the signals a fixed threshold cannot capture — cost per request suddenly running well above its baseline, or the quality score sliding — because “double yesterday’s cost” matters even if no absolute line was crossed. Route by **severity**: a real incident pages, a soft anomaly opens a ticket. The failure mode to fear is **alert fatigue**: page on every twitch and on-call learns to ignore the pager, so the one real incident gets muted with the noise. The block evaluates windows against an SLO and decides page-or-quiet, keyless.

> 🔔 **Analogy**
>
> It is a **smoke alarm that knows toast from a fire**. A good alarm ignores the momentary wisp when you make toast (the blip) but shrieks the instant there is real, sustained smoke (the incident). A cheap one goes off every time you cook — so within a week someone pulls the battery out, and now it is silent during the actual fire. That pulled battery is alert fatigue. The SLO is the smoke threshold; requiring several bad windows is the alarm waiting to be sure it is a fire and not toast.

Your p95 SLO is breached in one five-minute window, then recovers on its own. A good alerting rule does what?

**📝 `06_alerting.py`** - *runnable*

In [ ]:
# Alert on SYMPTOMS against an SLO, and stay quiet on a blip. A brief spike is not an incident.
SLO = {"p95_ms": 3000, "error_budget": 0.005, "cost_ratio_max": 1.5, "quality_floor": 7.5}

def evaluate(window):
    fires = []
    if window["p95_ms"] > SLO["p95_ms"]:
        fires.append(("p95 latency", "{}ms > {}ms SLO".format(window["p95_ms"], SLO["p95_ms"])))
    if window["error_rate"] > SLO["error_budget"]:
        fires.append(("error budget", "{:.1%} > {:.1%} budget".format(window["error_rate"], SLO["error_budget"])))
    if window["cost_ratio"] > SLO["cost_ratio_max"]:
        fires.append(("cost anomaly", "{:.1f}x baseline > {:.1f}x".format(window["cost_ratio"], SLO["cost_ratio_max"])))
    if window["quality"] < SLO["quality_floor"]:
        fires.append(("quality drop", "{:.1f} < {:.1f} floor".format(window["quality"], SLO["quality_floor"])))
    return fires

# A rule only PAGES if it breaches for 3 consecutive windows (a blip must not wake anyone).
def page_or_quiet(history):
    breached = [len(evaluate(w)) > 0 for w in history]
    return all(breached[-3:]) and len(breached) >= 3

NORMAL  = {"p95_ms": 2400, "error_rate": 0.002, "cost_ratio": 1.1, "quality": 8.3}
BLIP    = {"p95_ms": 4200, "error_rate": 0.004, "cost_ratio": 1.2, "quality": 8.1}
INCIDENT= {"p95_ms": 5200, "error_rate": 0.030, "cost_ratio": 1.9, "quality": 6.4}

print("Three windows, evaluated against the SLO:")
for label, w in [("normal", NORMAL), ("brief blip", BLIP), ("real incident", INCIDENT)]:
    fires = evaluate(w)
    tag = "OK" if not fires else ", ".join(n for n, _ in fires)
    print("  {:<14} -> {}".format(label, tag if fires else "all signals within SLO"))
print()
print("But WHEN to page (need 3 consecutive bad windows, or a blip wakes on-call for nothing):")
print("  normal, normal, blip           -> " + ("PAGE" if page_or_quiet([NORMAL, NORMAL, BLIP]) else "stay quiet"))
print("  blip, incident, incident, incident -> " + ("PAGE" if page_or_quiet([BLIP, INCIDENT, INCIDENT, INCIDENT]) else "stay quiet"))
print()
print("Page on sustained SLO breaches, log the blips. Every false page trains on-call to ignore the next.")

# Output:
# Three windows, evaluated against the SLO:
#   normal         -> all signals within SLO
#   brief blip     -> p95 latency
#   real incident  -> p95 latency, error budget, cost anomaly, quality drop
#
# But WHEN to page (need 3 consecutive bad windows, or a blip wakes on-call for nothing):
#   normal, normal, blip           -> stay quiet
#   blip, incident, incident, incident -> PAGE
#
# Page on sustained SLO breaches, log the blips. Every false page trains on-call to ignore the next.

- Each window is scored against the **SLO**: p95 latency, error budget, a cost-ratio ceiling, and a quality floor.
- A **normal** window is clean; a **blip** trips one signal; a **real incident** trips several at once.
- The **page-or-quiet** rule requires several consecutive bad windows — so the blip stays quiet and only the sustained incident pages.
- That is the whole discipline: page on sustained SLO breaches, log the blips — every false page trains on-call to ignore the next.

#### 💡 What just happened

⚡ What just happened? Alert against an SLO with an error budget, page only on a sustained breach (several bad windows), and add anomaly rules for cost and quality drifts a static threshold misses. Tradeoff: waiting for a sustained breach adds a little detection delay, in exchange for not drowning on-call in false pages. Alert fatigue is the failure mode. Next: the signal none of the classic pillars produce - live answer quality.

- Dials for p95, error budget, cost ratio, and quality against their SLO lines
- Feed a normal window (quiet), a brief blip (quiet), and a sustained incident (pages)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Online Eval, Drift, and PII

### Online Eval, Drift, and PII

Score a sample of live traffic with an eval to catch quality drift days before errors do, and redact PII before anything enters a span. This closes the loop: the CI eval gate from 12.6, now running continuously in production.

The last piece is the signal that makes this an *AI* observability stack and not just a fast web-app one: **online evaluation**. You already gated CI on an eval in 12.6; online eval is that same idea run continuously on a **sample of live traffic** — an LLM-as-judge (or a cheaper heuristic) scores sampled real responses, and you track the **rolling** quality over time. This is the only thing that catches a slow **drift**: a model swap or a creeping prompt change can pull quality down for days while the error rate stays at zero, and a rolling score crossing a floor is your early warning. Pair it with **data-quality** checks (input drift, a rising rate of empty or truncated outputs) and you see regressions before the tickets do. But there is a hard rule that comes with capturing live traffic: **redact PII first**. Prompts and responses carry emails, phone numbers, and secrets, and a trace ships them to a third-party backend — so scrub them *before* anything is written to a span, never after. The block runs a drift check and a redactor, keyless.

> 🔬 **Analogy**
>
> It is a **taste-tester on the live production line, with a shredder at the door**. The factory’s machines all report “running, on-spec” (errors zero), but only someone actually tasting a sample off the belt notices the recipe has drifted bland over the week — that is online eval catching quality drift. And the shredder is the PII rule: anything with a customer’s name or number on it gets scrubbed before it leaves the floor for the outside lab, because once it is in the vendor’s hands you cannot pull it back.

Error rate is flat at zero, but a model change three days ago quietly made answers worse. What surfaces this?

**📝 `07_online_eval.py`** - *runnable*

In [ ]:
# Online eval: score a sample of LIVE traffic and watch the trend for DRIFT.
# Separately: redact PII BEFORE anything enters a trace.
import re

# A rolling quality score over several days (an LLM judge on sampled live replies, 0-10).
daily_quality = [8.4, 8.3, 8.1, 7.6, 6.8, 6.2]
FLOOR = 7.0
window = 3   # alert if the rolling mean of the last 3 days falls below the floor

print("Online eval - rolling answer quality on sampled live traffic:")
for i in range(len(daily_quality)):
    lo = max(0, i - window + 1)
    roll = sum(daily_quality[lo:i + 1]) / (i - lo + 1)
    flag = "  <- DRIFT ALERT (below {} floor)".format(FLOOR) if roll < FLOOR else ""
    print("  day {}: score {:.1f}   rolling({}) {:.2f}{}".format(i + 1, daily_quality[i], window, roll, flag))
print("  quality drifted down for days before any error fired - only an eval loop catches this.")
print()

# PII must be redacted BEFORE the prompt/response is written into a span.
def redact(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]", text)
    text = re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "[PHONE]", text)
    return text

raw = "Refund to priya.n@example.com or call 555-018-2234 about order 7781"
print("PII redaction before capture:")
print("  raw prompt     : " + raw)
print("  stored in trace: " + redact(raw))
print("  the email and phone never touch your telemetry backend.")

# Output:
# Online eval - rolling answer quality on sampled live traffic:
#   day 1: score 8.4   rolling(3) 8.40
#   day 2: score 8.3   rolling(3) 8.35
#   day 3: score 8.1   rolling(3) 8.27
#   day 4: score 7.6   rolling(3) 8.00
#   day 5: score 6.8   rolling(3) 7.50
#   day 6: score 6.2   rolling(3) 6.87  <- DRIFT ALERT (below 7.0 floor)
#   quality drifted down for days before any error fired - only an eval loop catches this.
#
# PII redaction before capture:
#   raw prompt     : Refund to priya.n@example.com or call 555-018-2234 about order 7781
#   stored in trace: Refund to [EMAIL] or call [PHONE] about order 7781
#   the email and phone never touch your telemetry backend.

**📝 `otel-langfuse-instrumentation.py`** - *illustrative*

In [ ]:
# OTEL + LANGFUSE INSTRUMENTATION - an illustrative minimal example (needs the OTel SDK + a backend).
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from anthropic import Anthropic

# Point the OTLP exporter at ANY backend (Langfuse here; Phoenix / Datadog take the same span unchanged).
provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(
    endpoint="https://cloud.langfuse.com/api/public/otel/v1/traces")))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("ai-svc")
client = Anthropic()

def chat(prompt: str) -> str:
    # One CLIENT span tagged with the OTel gen_ai.* semantic conventions.
    with tracer.start_as_current_span("chat", kind=trace.SpanKind.CLIENT) as span:
        span.set_attribute("gen_ai.operation.name", "chat")
        span.set_attribute("gen_ai.system", "anthropic")
        span.set_attribute("gen_ai.request.model", "claude-sonnet-4-6")
        try:
            resp = client.messages.create(
                model="claude-sonnet-4-6", max_tokens=512,
                messages=[{"role": "user", "content": redact(prompt)}])   # redact PII BEFORE it enters the span
        except Exception as e:
            # capture the failure ON the span so the trace answers "why did THIS request fail?"
            span.set_status(Status(StatusCode.ERROR))
            span.record_exception(e)
            span.set_attribute("error.type", type(e).__name__)
            raise
        span.set_attribute("gen_ai.usage.input_tokens", resp.usage.input_tokens)
        span.set_attribute("gen_ai.usage.output_tokens", resp.usage.output_tokens)
        # end_turn is a clean finish; max_tokens or a refusal is a silent quality miss worth alerting on.
        span.set_attribute("gen_ai.response.finish_reasons", [resp.stop_reason])
        return resp.content[0].text
# Output: (illustrative minimal example - needs opentelemetry-sdk + an OTLP backend and a key; not run here.)

- The **rolling quality** score drifts down day over day; when the rolling average crosses the floor, a **drift alert** fires — days before any error would.
- **PII redaction** turns an email and phone number into `[EMAIL]` and `[PHONE]` *before* the text enters the trace — they never reach the backend.
- The **illustrative block** shows the real 2026 stack: an OTel `gen_ai` span exported over OTLP to Langfuse (or Phoenix / Datadog), with `redact()` applied before capture.
- The **error path** matters as much as the happy path: on an exception the span records the error and sets an **ERROR** status, and a non-`end_turn` finish reason flags a silent truncation or refusal — so the trace answers *why* a request failed, not just that it did.
- That instrumentation is the whole lesson in one function — a standard span, the golden-signal attributes, error capture, and PII scrubbed at the door.

#### 💡 What just happened

⚡ What just happened? Online eval scores sampled live traffic to catch quality drift days before errors do, and PII must be redacted before it enters a span. Tradeoff: sampling and judging live traffic costs a little compute and latency, in exchange for the one signal that catches a silent regression. This closes the loop from 12.6’s CI eval gate - now running continuously in production.

- A rolling quality line drifts down across days and crosses a floor into a DRIFT ALERT
- A prompt with an email and phone gets redacted before it enters the trace

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> An AI observability stack is a set of pieces, each answering a question the ops dashboard cannot. It starts by rejecting the green-dashboard illusion: an AI app fails by getting **worse**, not just by going down (Step 1). You capture behavior through the three pillars — **logs**, **metrics**, **traces** — plus **evals** as a fourth (Step 2), leaning on the **trace** to localize a slow request and surface **time-to-first-token** (Step 3). You emit those spans with **OpenTelemetry’s gen_ai conventions** so you instrument once and export anywhere (Step 4), put the **golden signals** — latency, TTFT, error rate, tokens, cost, cache, quality, sliced by model and team — on the dashboard (Step 5), and alert on **SLOs and anomalies**, paging only on a sustained breach (Step 6). Finally, **online eval** watches live quality to catch drift, with **PII redacted** before capture (Step 7). Ask five questions of any observability setup: does it **capture traces**, use **OpenTelemetry**, track the **AI signals** (cost, TTFT, quality), **alert on symptoms** not blips, and **watch quality in production**?

| Pillar | Question it answers | Shape | Reach for it when |
|---|---|---|---|
| Log | what happened in this one event? | one record, many fields | you need the details of a specific event |
| Metric | how is it trending across requests? | a number over time | you want dashboards and alerts |
| Trace | where did this request spend its time? | a tree of timed spans | you are debugging one slow request |
| Eval | was the answer any good? | a quality score over time | you are watching for quality drift |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now see an AI app the way it actually behaves. Using this telemetry to **diagnose and resolve an outage** is in Lesson 12.9; building the **eval set** your online-eval loop scores against — error analysis and eval-driven development — is in Lesson 14.4; and cutting the **cost** your dashboard surfaces is in Module 13. The primary references are the [OpenTelemetry gen_ai semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/), [OpenLLMetry](https://github.com/traceloop/openllmetry) for auto-instrumentation, [Langfuse](https://langfuse.com/docs) and [Arize Phoenix](https://github.com/Arize-ai/phoenix) for LLM tracing and evals, and [Prometheus](https://github.com/prometheus/prometheus) for the metrics pipeline.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Monitoring& Observability**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_8.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.8.html` - regenerate this notebook whenever the source HTML is updated.*
